In [1]:
import os
import pandas as pd

# =========================
# Config
# =========================
F_PATH  = "./results_F_37edges_100dags.csv"
OF_PATH = "./results_OF_orig_plus_47edges_100dags.csv"

OUT_DIR = "./mean"
OUT_F   = os.path.join(OUT_DIR, "mean_results_F_37edges.csv")
OUT_OF  = os.path.join(OUT_DIR, "mean_results_OF_orig_plus_47edges.csv")

# 평균낼 평가지표 컬럼들 (Score = 성능지표)
SCORE_COLS = ["AUROC", "AUPRC", "F1", "Brier", "ECE"]

# =========================
# Helpers
# =========================
def mean_scores_horizontal(df: pd.DataFrame, score_cols):
    # 컬럼 존재 여부 체크
    missing = [c for c in score_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing score columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    # 숫자 변환 (혹시 문자열 섞여 있어도 안전하게)
    df_score = df[score_cols].apply(pd.to_numeric, errors="coerce")

    mean_series = df_score.mean(axis=0, skipna=True)

    # 가로 1행 DataFrame
    mean_df = pd.DataFrame([mean_series.values], columns=mean_series.index)
    return mean_df


def main():
    # 1) 경로 체크
    for p in [F_PATH, OF_PATH]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"File not found: {p}")

    # 2) 출력 폴더 생성
    os.makedirs(OUT_DIR, exist_ok=True)

    # 3) 로드
    df_F  = pd.read_csv(F_PATH)
    df_OF = pd.read_csv(OF_PATH)

    # 4) 평균 계산
    mean_F  = mean_scores_horizontal(df_F,  SCORE_COLS)
    mean_OF = mean_scores_horizontal(df_OF, SCORE_COLS)

    # 5) 저장
    mean_F.to_csv(OUT_F, index=False)
    mean_OF.to_csv(OUT_OF, index=False)

    print(f"Saved: {OUT_F}  shape={mean_F.shape}")
    print(f"Saved: {OUT_OF} shape={mean_OF.shape}")


if __name__ == "__main__":
    main()


Saved: ./mean\mean_results_F_37edges.csv  shape=(1, 5)
Saved: ./mean\mean_results_OF_orig_plus_47edges.csv shape=(1, 5)
